#Who is the intended consumer of the sales report?

“The intended consumers of the sales report are sales managers, business analysts, finance teams, and senior management. They use the report to monitor sales performance, analyze revenue trends, and make informed business decisions.”

#Which raw tables are required to compute booked revenue?

“To calculate booked revenue, I need the Orders table for order status and date, and the Order Items table for quantity, unit price, and discount. If the business definition requires return or cancellation adjustments, I also need the relevant returns or cancellations table.”

#What is the output grain of the store report?

<!-- “Grain defines the level of detail represented by each row in a dataset. In PySpark, I use operations such as groupBy() to aggregate data to the required grain, such as order-level or store-level.” -->

“The output grain of the store report is one row per store, with sales metrics aggregated at the store level.”
Remember:
- Order-line grain: one row per order line.
- Order grain: one row per order.
- Store grain: one row per store.



#What is the output grain of the product report?

“The output grain of the product report is one row per product, with sales metrics aggregated at the product level.”

# Why must cancelled orders be excluded from booked revenue?

Cancelled orders must be excluded from booked revenue because the business does not expect to complete the sale. If an order is cancelled, the customer may receive a refund or the payment may never be collected. Therefore, including cancelled orders would overstate the actual booked revenue and produce inaccurate sales reports.

# What cleaning evidence should accompany a sales report?

“The sales report should include cleaning evidence such as null checks, duplicate removal, invalid value handling, cancelled-order exclusions, data type validation, input and output row counts, and revenue reconciliation. This evidence helps stakeholders trust the accuracy and reliability of the report.”

#Why should an input path be configurable?
“An input path should be configurable to make the pipeline reusable across development, testing, and production environments. It avoids hardcoding paths, reduces code changes, and makes maintenance easier.”

# What makes a report rerun reproducible?

“A report rerun is reproducible when the same input data, configuration, business rules, and code version consistently produce the same output. I ensure reproducibility through configurable paths, deterministic transformations, data validation, logging, and version control

#Why keep monetary outputs in Parquet rather than only screenshots?

“We store monetary outputs in Parquet because it preserves structured data and data types, supports efficient querying, and allows revenue totals to be validated and reconciled. Screenshots are useful as visual evidence, but they cannot replace reusable, machine-readable output files.”

#What business assumption must the report disclose about tax and shipping?
“The report must disclose whether tax and shipping charges are included in or excluded from the revenue calculation. If they are excluded, the report should explicitly state that net revenue is calculated using quantity multiplied by unit price minus discount, excluding tax and shipping charges. This ensures consistent interpretation and avoids misleading business decisions.”


In [ ]:
a=10
b=2
assert a > b
print("a is greater")

a is greater


In [ ]:
pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
spark=SparkSession.builder.appName("Retail Sales Report").getOrCreate()

# Hands-on questions (11-20)


In [ ]:
order = [
    (50001, 10001, 1001, 2, 45000.00, 5.0, None),
    (50002, 10001, 1005, 1, 12000.00, 10.0, 10800.00),
    (50003, 10002, 1003, 3, 2500.00, 5.0, 7125.00),
    (50004, 10002, 1008, -1, 18500.00, 15.0, 15725.00),
    (50005, 10003, 1002, 2, 32000.00, 8.0, 58880.00),
    (50006, 10003, 1006, 4, 1500.00, 5.0, 5700.00),
    (50007, 10004, 1004, None, 55000.00, 10.0, 49500.00),
    (50008, 10004, 1007, 2, 7500.00, None, 14250.00),
    (50009, 10005, 1009, 3, 2200.00, 10.0, 5940.00),
    (50010, 10006, 1010, 1, 48000.00, 12.0, 42240.00),
    (50011, 10007, 1003, 2, 2500.00, 5.0, 4750.00),
    (50012, 10008, 1005, 2, None, 7.5, 22200.00),
    (50013, 10009, 1001, 1, 45000.00, 10.0, 40500.00),
    (50014, 10010, 1006, 5, 1500.00, 5.0, 7125.00),
    (50015, 10011, 1008, 2, 18500.00, 10.0, 33300.00),
    (50016, 10012, 1002, -1, 32000.00, 15.0, 27200.00),
    (50017, 10013, 1004, 2, 55000.00, 5.0, 104500.00),
    (50018, 10014, 1007, 3, None, 8.0, 20700.00),
    (50019, 10015, 1009, 2, 2200.00, 5.0, 4180.00),
    (50020, 10015, 1010, 1, 48000.00, 10.0, 43200.00),
    (50021, 10021, 1021, None, 48000.00, 10.0, 43200.00),
    (50022, 10022, None, 1, 48000.00, 10.0, None)
]

order_schema=StructType([
    StructField("order_item_id",IntegerType(),True),
    StructField("order_id",IntegerType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DoubleType(),True),
    StructField("discount_pct",DoubleType(),True),
    StructField("item_amount",DoubleType(),True)
])

df_orders=spark.createDataFrame(order,order_schema)
df_orders.show()

+-------------+--------+----------+--------+----------+------------+-----------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_pct|item_amount|
+-------------+--------+----------+--------+----------+------------+-----------+
|        50001|   10001|      1001|       2|   45000.0|         5.0|       NULL|
|        50002|   10001|      1005|       1|   12000.0|        10.0|    10800.0|
|        50003|   10002|      1003|       3|    2500.0|         5.0|     7125.0|
|        50004|   10002|      1008|      -1|   18500.0|        15.0|    15725.0|
|        50005|   10003|      1002|       2|   32000.0|         8.0|    58880.0|
|        50006|   10003|      1006|       4|    1500.0|         5.0|     5700.0|
|        50007|   10004|      1004|    NULL|   55000.0|        10.0|    49500.0|
|        50008|   10004|      1007|       2|    7500.0|        NULL|    14250.0|
|        50009|   10005|      1009|       3|    2200.0|        10.0|     5940.0|
|        50010|   10006|    

In [ ]:
# Assemble ingestion, cleaning, deduplication, and aggregation into one script.
df_orders.write.csv("data.csv/orders.csv",header=True,mode="overwrite")
#ingestion
df_read=spark.read.option("header","true").schema(order_schema).csv("data.csv/orders.csv")
# df_read.show()
# cleaning
df_clean=df_read.filter((col("product_id").isNotNull()) & (col("quantity").isNotNull()) & (col("quantity")>0) & (col("unit_price").isNotNull()) & (col("discount_pct").isNotNull()) & (col("item_amount").isNotNull()))
# df_clean.show()

#deduplication
business_key=["order_item_id"]
df_deduplication=df_clean.dropDuplicates(business_key)
# df_deduplication.show()

# aggregation
df_aggregation=df_deduplication.withColumn("revenue",col("quantity")* col("unit_price")*(1-(col("discount_pct")/100)))
df_aggregation.show()
df_product_revenue=df_aggregation.groupBy("product_id").agg(sum("revenue").alias("product_revenue"))
df_product_revenue.show()



+-------------+--------+----------+--------+----------+------------+-----------+--------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_pct|item_amount| revenue|
+-------------+--------+----------+--------+----------+------------+-----------+--------+
|        50011|   10007|      1003|       2|    2500.0|         5.0|     4750.0|  4750.0|
|        50003|   10002|      1003|       3|    2500.0|         5.0|     7125.0|  7125.0|
|        50002|   10001|      1005|       1|   12000.0|        10.0|    10800.0| 10800.0|
|        50006|   10003|      1006|       4|    1500.0|         5.0|     5700.0|  5700.0|
|        50009|   10005|      1009|       3|    2200.0|        10.0|     5940.0|  5940.0|
|        50010|   10006|      1010|       1|   48000.0|        12.0|    42240.0| 42240.0|
|        50005|   10003|      1002|       2|   32000.0|         8.0|    58880.0| 58880.0|
|        50014|   10010|      1006|       5|    1500.0|         5.0|     7125.0|  7125.0|
|        5

In [ ]:
from decimal import Decimal

In [ ]:
store_revenue = [

    (2001, 101, 301, "Laptop",          "Electronics", 2, Decimal("65000.00"), Decimal("1500.00"), Decimal("52000.00")),
    (2001, 101, 301, "Laptop",          "Electronics", 2, Decimal("65000.00"), Decimal("1500.00"), Decimal("52000.00")),
    (2002, 101, 302, "Wireless Mouse",  "Accessories", 5, Decimal("750.50"),   Decimal("50.00"),   Decimal("500.00")),
    (2003, 102, 303, "Office Chair",    "Furniture",   3, Decimal("5500.00"),  Decimal("300.00"),  Decimal("4000.00")),
    (2004, 103, 304, "Desk Lamp",       "Lighting",    4, Decimal("1250.75"), Decimal("100.00"),  Decimal("850.00")),
    (2005, 101, 301, "Laptop",          "Electronics", 1, Decimal("65000.00"), Decimal("2000.00"), Decimal("52000.00")),
    (2006, 104, 305, "Keyboard",        "Accessories", 6, Decimal("1800.00"),  Decimal("150.00"),  Decimal("1200.00")),
    (2007, 102, 306, "Monitor",         "Electronics", 2, Decimal("18500.00"), Decimal("500.00"),  Decimal("14500.00")),
    (2008, 103, 307, None,              "Furniture",   2, Decimal("3200.00"),  Decimal("100.00"),  Decimal("2400.00")),
    (2009, 104, 302, "Wireless Mouse",  "Accessories", 8, Decimal("750.50"),   Decimal("75.00"),   Decimal("500.00")),
    (2010, 105, 308, "Headphones",      "Electronics", 3, Decimal("2500.00"),  Decimal("200.00"),  Decimal("1700.00")),
    (2011, 101, 309, "USB Hub",         "Accessories", 5, Decimal("899.99"),   None,               Decimal("600.00")),
    (2012, 102, 310, "Bookshelf",       "Furniture",   None, Decimal("7500.00"), Decimal("500.00"), Decimal("5500.00")),
    (2013, 103, 304, "Desk Lamp",       "Lighting",    4, Decimal("1250.75"), Decimal("100.00"),  Decimal("850.00")),
    (2014, 104, 311, "Smartphone",      "Electronics", 2, Decimal("42000.00"), Decimal("1000.00"), Decimal("35000.00")),
    (2015, 105, 312, "Printer",         "Electronics", 1, Decimal("12500.00"), Decimal("750.00"), Decimal("9800.00")),
]

schema=StructType([
    StructField("sale_id",IntegerType(),False),
    StructField("store_id",IntegerType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("product_name",StringType(),True),
    StructField("category",StringType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DecimalType(10,2),True),
    StructField("discount",DecimalType(10,2),True),
    StructField("cost_price",DecimalType())
])
df_store=spark.createDataFrame(store_revenue,schema)

In [ ]:
# Publish revenue and AOV by store.
# df_store.show()
df_clean=df_store.filter((col("product_name").isNotNull()) & (col("quantity").isNotNull()) & col("discount").isNotNull())
# df_clean.show()
df_distinct=df_clean.dropDuplicates(["sale_id"])
df_revenue=df_distinct.withColumn("revenue",col("quantity")*col("unit_price")-col("discount"))
df_revenue.show()
# AOV by store
# df_aov=df_revenue.groupBy("store_id").agg(sum("revenue")/F.count("store_id")).alias("aov")
df_aov=df_revenue.groupBy("store_id").agg((avg("revenue")).alias("store_average"))
df_aov.show()


+-------+--------+----------+--------------+-----------+--------+----------+--------+----------+---------+
|sale_id|store_id|product_id|  product_name|   category|quantity|unit_price|discount|cost_price|  revenue|
+-------+--------+----------+--------------+-----------+--------+----------+--------+----------+---------+
|   2001|     101|       301|        Laptop|Electronics|       2|  65000.00| 1500.00|     52000|128500.00|
|   2002|     101|       302|Wireless Mouse|Accessories|       5|    750.50|   50.00|       500|  3702.50|
|   2003|     102|       303|  Office Chair|  Furniture|       3|   5500.00|  300.00|      4000| 16200.00|
|   2004|     103|       304|     Desk Lamp|   Lighting|       4|   1250.75|  100.00|       850|  4903.00|
|   2005|     101|       301|        Laptop|Electronics|       1|  65000.00| 2000.00|     52000| 63000.00|
|   2006|     104|       305|      Keyboard|Accessories|       6|   1800.00|  150.00|      1200| 10650.00|
|   2007|     102|       306|       M

In [ ]:
store_order = [
    (5001, 101, "2026-09-20", "Completed", "UPI", "Visakhapatnam", "Andhra Pradesh", 12500.50, 750.00, 100.00, "Website", 201),
    (5002, 102, "2026-09-18", "Completed", "Card", "Hyderabad", "Telangana", 8500.00, 500.00, 150.00, "Mobile App", 202),
    (5003, 103, "2026-09-15", "Pending", "NetBanking", "Bengaluru", "Karnataka", 6200.75, 300.00, 120.00, "Marketplace", 203),
    (5004, 104, "2026-09-12", "Completed", "Card", "Mumbai", "Maharashtra", 18500.00, 1200.00, 200.00, "Website", 204),
    (5005, 105, "2026-09-10", "Cancelled", "UPI", "Chennai", "Tamil Nadu", 4500.25, 250.00, 100.00, "Physical Store", 205),
    (5006, 106, "2026-09-08", "Completed", "Cash", "Vijayawada", "Andhra Pradesh", 3200.00, 150.00, 80.00, "Physical Store", 206),
    (5007, 107, "2026-09-05", "Returned", "Card", "Pune", "Maharashtra", 9800.50, 600.00, 150.00, "Marketplace", 201),
    (5008, 108, "2026-09-02", "Completed", "NetBanking", "Delhi", "Delhi", 15200.00, 1000.00, 180.00, "Mobile App", 208),
    (5009, 109, "2026-08-28", "Pending", "UPI", "Kolkata", "West Bengal", 2750.75, 100.00, 70.00, "Website", 202),
    (5010, 110, "2026-08-25", "Completed", "Card", "Ahmedabad", "Gujarat", 11200.00, 800.00, 150.00, "Marketplace", 210),
    (5011, 111, "2026-08-20", "Completed", "UPI", "Kochi", "Kerala", 7350.50, 450.00, 120.00, "Website", 211),
    (5012, 112, "2026-08-15", "Cancelled", "Cash", "Bhubaneswar", "Odisha", 3900.00, 200.00, 90.00, "Physical Store", 212),
    (5013, 113, "2026-08-10", "Completed", "NetBanking", "Coimbatore", "Tamil Nadu", 16400.25, 950.00, 175.00, "Mobile App", 213),
    (5014, 114, "2026-08-05", "Returned", "Card", "Tirupati", "Andhra Pradesh", 5600.00, 350.00, 100.00, "Marketplace", 214),
    (5015, 115, "2026-08-01", "Completed", "UPI", "Jaipur", "Rajasthan", 8900.75, 550.00, 130.00, "Website", 215)
]
columns = [
    "order_id", "customer_id", "order_date", "booking_status",
    "payment_method", "city", "state", "booking_amount",
    "discount", "shipping_fee", "sales_channel", "store_id"
]

df_store_order=spark.createDataFrame(store_order,columns)

In [ ]:
# Publish revenue and revenue share by channel.
# df_store.show()
df_filter=df_store_order.filter(col("booking_status")=="Completed")
# df_filter.show()
df_revenue=df_filter.withColumn("revenue",col("booking_amount")-col("discount"))
# df_revenue.show()
df_channel_revenue=df_revenue.groupBy("sales_channel").agg(sum("revenue").alias("channel_revenue"))
# df_channel_revenue.show()
df_total_revenue=df_channel_revenue.agg(sum("channel_revenue").alias("total_revenue"))
# df_total_revenue.show()
df_join=df_channel_revenue.join(df_total_revenue,how="cross")
df_join.show()
df_revenue_share=df_join.withColumn("revenue_share",(col("channel_revenue")/col("total_revenue"))*100)
df_revenue_share.show()


+--------------+---------------+-------------+
| sales_channel|channel_revenue|total_revenue|
+--------------+---------------+-------------+
|    Mobile App|       37650.25|      95402.0|
|Physical Store|         3050.0|      95402.0|
|       Website|       44301.75|      95402.0|
|   Marketplace|        10400.0|      95402.0|
+--------------+---------------+-------------+

+--------------+---------------+-------------+------------------+
| sales_channel|channel_revenue|total_revenue|     revenue_share|
+--------------+---------------+-------------+------------------+
|    Mobile App|       37650.25|      95402.0| 39.46484350432905|
|Physical Store|         3050.0|      95402.0| 3.196997966499654|
|       Website|       44301.75|      95402.0| 46.43691956143477|
|   Marketplace|        10400.0|      95402.0|10.901238967736525|
+--------------+---------------+-------------+------------------+



In [ ]:
df_store.show()

+-------+--------+----------+--------------+-----------+--------+----------+--------+----------+
|sale_id|store_id|product_id|  product_name|   category|quantity|unit_price|discount|cost_price|
+-------+--------+----------+--------------+-----------+--------+----------+--------+----------+
|   2001|     101|       301|        Laptop|Electronics|       2|  65000.00| 1500.00|     52000|
|   2001|     101|       301|        Laptop|Electronics|       2|  65000.00| 1500.00|     52000|
|   2002|     101|       302|Wireless Mouse|Accessories|       5|    750.50|   50.00|       500|
|   2003|     102|       303|  Office Chair|  Furniture|       3|   5500.00|  300.00|      4000|
|   2004|     103|       304|     Desk Lamp|   Lighting|       4|   1250.75|  100.00|       850|
|   2005|     101|       301|        Laptop|Electronics|       1|  65000.00| 2000.00|     52000|
|   2006|     104|       305|      Keyboard|Accessories|       6|   1800.00|  150.00|      1200|
|   2007|     102|       306| 

In [ ]:
from pyspark.sql import Window

In [ ]:
# Publish the top ten products by booked revenue.
df_filter=df_store.filter((col("quantity").isNotNull()) & (col("unit_price").isNotNull()) & (col("discount").isNotNull()) & col("product_name").isNotNull())
# df_filter.show()
df_revenue=df_filter.withColumn("revenue",col("quantity")*col("unit_price")-col("discount"))
# df_revenue.show()
df_grouping=df_revenue.groupBy("product_name").agg(sum("revenue").alias("total_revenue")).orderBy(col("total_revenue").desc())
df_grouping.limit(10).show()


+--------------+-------------+
|  product_name|total_revenue|
+--------------+-------------+
|        Laptop|    320000.00|
|    Smartphone|     83000.00|
|       Monitor|     36500.00|
|  Office Chair|     16200.00|
|       Printer|     11750.00|
|      Keyboard|     10650.00|
|     Desk Lamp|      9806.00|
|Wireless Mouse|      9631.50|
|    Headphones|      7300.00|
+--------------+-------------+



In [ ]:
df_orders.show()

+-------------+--------+----------+--------+----------+------------+-----------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_pct|item_amount|
+-------------+--------+----------+--------+----------+------------+-----------+
|        50001|   10001|      1001|       2|   45000.0|         5.0|       NULL|
|        50002|   10001|      1005|       1|   12000.0|        10.0|    10800.0|
|        50003|   10002|      1003|       3|    2500.0|         5.0|     7125.0|
|        50004|   10002|      1008|      -1|   18500.0|        15.0|    15725.0|
|        50005|   10003|      1002|       2|   32000.0|         8.0|    58880.0|
|        50006|   10003|      1006|       4|    1500.0|         5.0|     5700.0|
|        50007|   10004|      1004|    NULL|   55000.0|        10.0|    49500.0|
|        50008|   10004|      1007|       2|    7500.0|        NULL|    14250.0|
|        50009|   10005|      1009|       3|    2200.0|        10.0|     5940.0|
|        50010|   10006|    

In [ ]:
#Publish raw, quarantined, removed-duplicate, and accepted counts.
# raw count
# df_orders.show()
raw_count=df_orders.count()
print("Raw Count:",raw_count)

# quarantined count
quarentine=df_orders.filter((col("quantity").isNull()) | (col("quantity")<=0) | (col("unit_price").isNull()) | (col("discount_pct").isNull()) | (col("item_amount").isNull()))
# quarentine.show()
quarentine_count=quarentine.count()
print("Quarentine Count:",quarentine_count)

#accepted counts
accepted=df_orders.filter((col("quantity").isNotNull()) & (col("unit_price").isNotNull()) & (col("quantity")>0) & (col("discount_pct").isNotNull()) & (col("item_amount").isNotNull()))
# unique_accept=accepted.distinct()
accpt_count=accepted.count()
# accepted.show()

#  removed-duplicate
unique_orders = accepted.dropDuplicates(["order_id"])
# normal_duplicates.show()
remove_duplicate=unique_orders.count()
remove_duplicates_count=accpt_count - remove_duplicate
print("Remove Duplicates Count:",remove_duplicates_count)
print("Accepted Count:",unique_orders.count())

# 6. Reconciliation check
print("Reconcilation Passed:",raw_count == quarentine_count+ unique_orders.count()+remove_duplicates_count)



Raw Count: 22
Quarentine Count: 9
Remove Duplicates Count: 2
Accepted Count: 11
Reconcilation Passed: True


In [ ]:
df_store_order.show()

+--------+-----------+----------+--------------+--------------+-------------+--------------+--------------+--------+------------+--------------+--------+
|order_id|customer_id|order_date|booking_status|payment_method|         city|         state|booking_amount|discount|shipping_fee| sales_channel|store_id|
+--------+-----------+----------+--------------+--------------+-------------+--------------+--------------+--------+------------+--------------+--------+
|    5001|        101|2026-09-20|     Completed|           UPI|Visakhapatnam|Andhra Pradesh|       12500.5|   750.0|       100.0|       Website|     201|
|    5002|        102|2026-09-18|     Completed|          Card|    Hyderabad|     Telangana|        8500.0|   500.0|       150.0|    Mobile App|     202|
|    5003|        103|2026-09-15|       Pending|    NetBanking|    Bengaluru|     Karnataka|       6200.75|   300.0|       120.0|   Marketplace|     203|
|    5004|        104|2026-09-12|     Completed|          Card|       Mumbai

In [ ]:
# Verify booked revenue equals the manifest's expected amount.
complete=df_store_order.filter(col("booking_status")=="Completed")
# complete.show()
df_filter = complete.withColumn("revenue",col("booking_amount")-col("discount"))
df_filter=df_filter.select("order_id","revenue")
# df_filter.show()

revenue_manifest = [
    (5001, 11750.50),
    (5002, 8000.00),
    (5004, 17300.00),
    (5006, 3050.00),
    (5008, 14200.00),
    (5010, 10400.10),
    (5011, 6900.50),
    (5013, 15450.25),
    (5015, 8350.70)
]
cols=["order_id","manifest_revenue"]
df_manifest=spark.createDataFrame(revenue_manifest,cols)
df_join=df_filter.join(df_manifest,on="order_id",how="inner")
df_join.show()

df_check=df_join.withColumn("check",when(col("revenue")==col("manifest_revenue"),"Reconcilation Pass").otherwise("Reconcilation Failed"))
df_check.show()


+--------+--------+----------------+
|order_id| revenue|manifest_revenue|
+--------+--------+----------------+
|    5001| 11750.5|         11750.5|
|    5002|  8000.0|          8000.0|
|    5004| 17300.0|         17300.0|
|    5006|  3050.0|          3050.0|
|    5008| 14200.0|         14200.0|
|    5010| 10400.0|         10400.1|
|    5011|  6900.5|          6900.5|
|    5013|15450.25|        15450.25|
|    5015| 8350.75|          8350.7|
+--------+--------+----------------+

+--------+--------+----------------+--------------------+
|order_id| revenue|manifest_revenue|               check|
+--------+--------+----------------+--------------------+
|    5001| 11750.5|         11750.5|  Reconcilation Pass|
|    5002|  8000.0|          8000.0|  Reconcilation Pass|
|    5004| 17300.0|         17300.0|  Reconcilation Pass|
|    5006|  3050.0|          3050.0|  Reconcilation Pass|
|    5008| 14200.0|         14200.0|  Reconcilation Pass|
|    5010| 10400.0|         10400.1|Reconcilation Fail

In [ ]:
# Write a short explanation of the best-performing store.
df_store_order.show()

+--------+-----------+----------+--------------+--------------+-------------+--------------+--------------+--------+------------+--------------+--------+
|order_id|customer_id|order_date|booking_status|payment_method|         city|         state|booking_amount|discount|shipping_fee| sales_channel|store_id|
+--------+-----------+----------+--------------+--------------+-------------+--------------+--------------+--------+------------+--------------+--------+
|    5001|        101|2026-09-20|     Completed|           UPI|Visakhapatnam|Andhra Pradesh|       12500.5|   750.0|       100.0|       Website|     201|
|    5002|        102|2026-09-18|     Completed|          Card|    Hyderabad|     Telangana|        8500.0|   500.0|       150.0|    Mobile App|     202|
|    5003|        103|2026-09-15|       Pending|    NetBanking|    Bengaluru|     Karnataka|       6200.75|   300.0|       120.0|   Marketplace|     203|
|    5004|        104|2026-09-12|     Completed|          Card|       Mumbai